# JiT-S2-Spatial-Mamba — no gate + **3 in-context class tokens** inside the scan

The no-gate Spatial-Mamba run (FID 93.95) is conditioned by adaLN-Zero only. In JiM (VMamba), adding **one** class token at the head of the scan was the best conditioning route (class m = 1: **89.20** vs 91.73). Spatial-Mamba has a single row-major scan, so a head token is far from most of the image; SASF helps, because it reads states up to 3 rows / columns away in every direction. This run places **3 class tokens inside the scan**, each just before grid cell **(0, 3), (3, 3), (7, 3)** (row, col on the 8×8 grid).

Scan steps from the nearest class token, per layer (max / mean over the 64 image tokens):

| placement | max | mean |
|---|---|---|
| 1 head token, no SASF | 64 | 32.5 |
| 1 head token, with SASF | 37 | 12.25 |
| VMamba, 4 scans, 1 head token (FID 89.20) | 29 | 11.5 |
| **this: 3 tokens at (0,3), (3,3), (7,3), with SASF** | **2** | **1.19** |

(0, 3), (3, 3), (7, 3) is the best of every choice of 3 rows and column; column 3 lets the cells left of a token read the fused state just after it. Scan length 64 → 67 (+4.7%).

- Same scheme as the VMamba in-context arm: tokens `y_emb + slot` prepended once at **block 4**, updated by blocks 4–11, stripped before the final layer. adaLN-Zero stays on.
- The tokens share `in_proj`, skip the mixer's DWConv and get only the LPU's centre tap (no grid cell). SASF fuses the grid states only.
- Everything else as the no-gate run: d_state 1, expand 1, **no gate**, LPU (zero-init), dilations [1, 2, 3].

Model: **29,687,268** params (no-gate run + 3 × 384 slots).

| model (Tiny-ImageNet, 100 epochs) | params | FID-10K |
|---|---|---|
| JiM baseline: VMamba, d_state 16 | 31.03M | 91.73 |
| JiM + 1 class token (head) | 31.03M | **89.20** |
| Spatial, no gate, d_state 1 + LPU | 29.69M | 93.95 |
| **this:** + 3 class tokens at (0,3), (3,3), (7,3) | 29.69M | ? |

The verify cell proves the arm on CPU (off = bit-identical to the finished runs; on = a naive float64 transcription) and prints this run's it/s and peak memory next to the no-gate run.

---

JiT-S2 on Tiny-ImageNet-200 (64px, patch 8, 200 classes) via the repo's `run_experiment.py` / `evaluate.py`. Comment out the download cell if you attach a dataset instead, and the eval cell if you only want to train.

## 1. Environment  *(Internet ON)*

In [ ]:
import os, sys

# Pick the stack by the image's Python. torch 2.5.1 has no matching torchvision
# for Python 3.13 (Kaggle's newer image), so 3.13 moves to torch 2.6.0; 3.12
# keeps torch 2.5.1 exactly as every earlier run. mamba-ssm 2.2.4 and
# causal-conv1d 1.5.0.post8 publish cp312 and cp313 wheels for both.
PY = f"cp{sys.version_info.major}{sys.version_info.minor}"
if sys.version_info >= (3, 13):
    TORCH, TVISION, TAUDIO, TTAG = "2.6.0", "0.21.0", "2.6.0", "torch2.6"
else:
    TORCH, TVISION, TAUDIO, TTAG = "2.5.1", "0.20.1", "2.5.1", "torch2.5"
print("python", sys.version.split()[0], "->", PY, "| torch", TORCH)

# 1) Pin torch
!pip install -q torch=={TORCH} torchvision=={TVISION} torchaudio=={TAUDIO} \
    --index-url https://download.pytorch.org/whl/cu124

# 2) Download wheels with explicit destination
CAUSAL = f"causal_conv1d-1.5.0.post8+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"
MAMBA  = f"mamba_ssm-2.2.4+cu12{TTAG}cxx11abiFALSE-{PY}-{PY}-linux_x86_64.whl"

os.system(f"wget -q https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.5.0.post8/{CAUSAL} -O /kaggle/working/{CAUSAL}")
os.system(f"wget -q https://github.com/state-spaces/mamba/releases/download/v2.2.4/{MAMBA} -O /kaggle/working/{MAMBA}")
for w in (CAUSAL, MAMBA):
    p = f"/kaggle/working/{w}"
    assert os.path.exists(p) and os.path.getsize(p) > 0, f"wheel download failed: {w}"

!pip install -q /kaggle/working/{CAUSAL}
!pip install -q /kaggle/working/{MAMBA}

# 3) Patch mamba-ssm
import glob
for path in glob.glob("/usr/local/lib/python*/dist-packages/mamba_ssm/utils/generation.py"):
    with open(path) as f: src = f.read()
    new = src.replace(
        "from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer",
        "from transformers.generation import GenerateDecoderOnlyOutput, TextStreamer",
    ).replace(
        "output_cls = GreedySearchDecoderOnlyOutput if top_k == 1 else SampleDecoderOnlyOutput",
        "output_cls = GenerateDecoderOnlyOutput",
    )
    if new != src:
        with open(path, "w") as f: f.write(new)
        print(f"\u2705 Patched {path}")

print(">>> RESTART RUNTIME NOW <<<")

## 2. Repo  *(clone + cd)*

In [ ]:
# Clone the repo. The models, configs, wiring, CPU tests and the GPU check are on main.
import os
REPO_DIR = "/kaggle/working/thesis_Choustoulakis"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Rodamanthosch/thesis_Choustoulakis.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# sanity: the in-context token placement is wired into both entry points
!grep -q 'in_context_positions' scripts/run_experiment.py && grep -q 'in_context_positions' scripts/evaluate.py \
    && echo "wiring OK (train + eval)" || echo "WIRING MISSING -- pull latest main"

## 3. Verify the model  *(first session only; ~3-6 min)*

In [ ]:
# === Verify the model (~3-5 min; first session only -- comment out on resume).
# PYTHONPATH=. is REQUIRED: without it "from src.models..." fails and the
# tests' try/except misreports it as a missing mamba_ssm (known issue).
# If anything FAILS, STOP -- do not train a model that is not the one reported.
#  CPU proofs: the Spatial model is unchanged (all its checks); with the tokens
#  OFF it is bit-identical to the finished runs; with them ON the mixer equals a
#  naive float64 transcription (scan order, SASF on the grid only, unfused
#  tokens), placement is causal, grad checkpointing is exact, the config's
#  distance figures hold, and both build_models agree on the config.
!PYTHONPATH=. python tests/test_spatial_mamba_cpu.py
!PYTHONPATH=. python tests/test_spatial_incontext_cpu.py
#  GPU check (scripts/check_spatial_incontext_cuda.py): the mixer with tokens on
#  CUDA matches the CPU reference, the config trains under bf16 with grads into
#  the token slots, and it/s + peak memory at batch 128 vs the no-gate run.
!PYTHONPATH=. python scripts/check_spatial_incontext_cuda.py

## 4. Tiny-ImageNet  *(attached dataset, else download to /tmp)*

In [ ]:
# Tiny-ImageNet-200 -> /tmp (EPHEMERAL), ImageFolder layout for train/ and val/.
# Sources, in order: an attached Kaggle dataset containing tiny-imagenet-200/,
# then Stanford (https, http), then the SJTU mirror of the identical zip.
import os, zipfile, glob, shutil, subprocess, hashlib
DATA_DIR = "/tmp/tiny-imagenet-200"
ZIP      = "/tmp/tiny-imagenet-200.zip"
MD5      = "90528d7ca1a48142e341f4ef8d21d0de"
SOURCES  = ["https://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "http://cs231n.stanford.edu/tiny-imagenet-200.zip",
            "https://scidata.sjtu.edu.cn/records/swpcp-01e87/files/tiny-imagenet-200.zip?download=1"]

def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    attached = [os.path.dirname(p) for p in
                glob.glob("/kaggle/input/**/tiny-imagenet-200/train", recursive=True)]
    if attached:
        print("copying attached dataset:", attached[0])
        shutil.copytree(attached[0], DATA_DIR)
    else:
        if os.path.exists(ZIP) and (not zipfile.is_zipfile(ZIP) or _md5(ZIP) != MD5):
            print(f"removing broken download ({os.path.getsize(ZIP)} bytes)")
            os.remove(ZIP)
        for src in SOURCES:
            if os.path.exists(ZIP):
                break
            print("downloading", src, "(~248MB)...")
            r = subprocess.run(["wget", "-q", "--tries=3", "--timeout=60", "-O", ZIP, src])
            ok = r.returncode == 0 and zipfile.is_zipfile(ZIP) and _md5(ZIP) == MD5
            if not ok:
                size = os.path.getsize(ZIP) if os.path.exists(ZIP) else 0
                print(f"  failed (wget exit {r.returncode}, {size} bytes)")
                if os.path.exists(ZIP):
                    os.remove(ZIP)
        assert os.path.exists(ZIP), ("Tiny-ImageNet download failed from every source -- "
                                     "attach a Kaggle tiny-imagenet dataset (Add Input) and rerun this cell")
        print("extracting...")
        with zipfile.ZipFile(ZIP) as z:
            z.extractall("/tmp")

# TRAIN: train/<cls>/images/*.JPEG -> train/<cls>/*.JPEG  (ImageFolder layout)
tr = os.path.join(DATA_DIR, "train")
for cls in os.listdir(tr):
    sub = os.path.join(tr, cls, "images")
    if os.path.isdir(sub):
        for f in glob.glob(os.path.join(sub, "*.JPEG")):
            shutil.move(f, os.path.join(tr, cls))
        shutil.rmtree(sub)
    for b in glob.glob(os.path.join(tr, cls, "*_boxes.txt")):
        os.remove(b)

# VAL: flat val/images + val_annotations.txt -> val/<cls>/*.JPEG  (evaluate.py reads val/)
val = os.path.join(DATA_DIR, "val")
ann = os.path.join(val, "val_annotations.txt")
if os.path.exists(ann):
    with open(ann) as f:
        for line in f:
            img, cls = line.split("\t")[:2]
            os.makedirs(os.path.join(val, cls), exist_ok=True)
            s = os.path.join(val, "images", img)
            if os.path.exists(s):
                shutil.move(s, os.path.join(val, cls, img))
    shutil.rmtree(os.path.join(val, "images"), ignore_errors=True)
    os.remove(ann)

n_tr, n_val = len(glob.glob(tr + "/*")), len(glob.glob(val + "/*"))
print("train classes:", n_tr, "| val classes:", n_val, "| DATA_DIR:", DATA_DIR)
assert n_tr == 200 and n_val == 200, "unexpected class count -- dataset incomplete"

## 5. Settings  *(edits the cloned config; finds the checkpoint to resume)*

In [ ]:
# ── Settings for THIS notebook ─────────────────────────────────────────
EPOCHS     = 100          # full target; resume across sessions until reached
SAVE_FREQ  = 1            # overwrite checkpoint-last.pt EVERY epoch (12h safety)

# Fallback if the download cell was commented out (attached-dataset workflow):
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/kaggle/input/tiny-imagenet/tiny-imagenet-200"  # <-- your attached dataset
    print("DATA_DIR not set by a download cell; using:", DATA_DIR)

CONFIG  = "configs/tiny_imagenet/jit-s2-spatial-nogate-incontext-class3.yaml"
RUN     = "exp_spatial_nogate_incontext_class3"
OUT_DIR = f"/kaggle/working/{RUN}"   # persists in the saved Version

# Sync the cloned config so BOTH training and evaluation read identical values
# (evaluate.py has no model CLI overrides -- it builds the model straight from
# config). adaLN-Zero + 3 in-context class tokens inside the scan.
import re
def set_cfg(path, kv):
    s = open(path).read()
    for k, v in kv.items():
        s, n = re.subn(rf"(?m)^(\s*{k}:).*$", rf"\1 {v}", s, count=1)
        assert n == 1, f"key {k!r} not found in {path}"
    open(path, "w").write(s)
set_cfg(CONFIG, {"data_dir": DATA_DIR, "gate": "false", "d_state": 1, "expand": 1,
                 "sasf_dilations": "[1, 2, 3]", "lpu": "zero",
                 "in_context_len": 3, "in_context_start": 4, "in_context_content": "class",
                 "in_context_positions": "[[0, 3], [3, 3], [7, 3]]",
                 "ffn": "swiglu"})

# ── Resume across sessions AND accounts (automatic) ────────────────────
# Looks for THIS model's newest checkpoint in: this session's OUT_DIR, any
# attached notebook output (.../checkpoint-last.pt), and any attached DATASET
# holding an uploaded checkpoint (Kaggle unpacks an uploaded .pt into a folder
# with data.pkl / data/ / byteorder / version -- it is zipped back here).
# A checkpoint is used only if its weights match this config's model exactly
# (same names and shapes), so another run's checkpoint is never picked up.
# The newest epoch wins. To force one, set RESUME_CKPT_FORCE to its path.
RESUME_CKPT_FORCE = None
import os, glob, zipfile, torch, yaml
from scripts.evaluate import build_model
_ref = {k: tuple(v.shape) for k, v in build_model(yaml.safe_load(open(CONFIG))).state_dict().items()}

def _rebuild(folder):
    """Zip an unpacked checkpoint folder back into a loadable .pt in /tmp."""
    dst = os.path.join("/tmp/resume", folder.strip("/").replace("/", "__") + ".pt")
    if not os.path.exists(dst):
        os.makedirs("/tmp/resume", exist_ok=True)
        with zipfile.ZipFile(dst, "w", zipfile.ZIP_STORED) as z:
            for root, _, files in os.walk(folder):
                for f in files:
                    full = os.path.join(root, f)
                    z.write(full, os.path.join("checkpoint", os.path.relpath(full, folder)))
    return dst

cands = [os.path.join(OUT_DIR, "checkpoint-last.pt")]
cands += sorted(glob.glob("/kaggle/input/**/checkpoint-last.pt", recursive=True))
cands += [_rebuild(os.path.dirname(p))
          for p in sorted(glob.glob("/kaggle/input/**/data.pkl", recursive=True))]
best = None
print("checkpoints found:")
for c in ([RESUME_CKPT_FORCE] if RESUME_CKPT_FORCE else cands):
    if not (c and os.path.exists(c)):
        continue
    try:
        p = torch.load(c, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  unreadable ({type(e).__name__}): {c}")
        continue
    match = {k: tuple(v.shape) for k, v in p.get("model", {}).items()} == _ref
    print(f"  {'THIS model ' if match else 'other model'}  epoch {p.get('epoch')}  {c}")
    if match and (best is None or p["epoch"] > best[0]):
        best = (p["epoch"], c)
    del p
RESUME_CKPT = best[1] if best else None

print("config :", CONFIG, "|", "spatial_mamba no gate, d_state 1, lpu zero, 3 class tokens before cells (0,3) (3,3) (7,3) from block 4")
print("data   :", DATA_DIR, "(exists:", os.path.isdir(DATA_DIR), ")")
print("out    :", OUT_DIR)
print("resume :", f"{RESUME_CKPT} (last finished epoch {best[0]})" if best else "(fresh start)")

## 6. Train  *(Save Version before 12h; next session it resumes by itself)*

In [ ]:
# ── TRAIN ──────────────────────────────────────────────────────────────
# data_dir + the model knobs now live in the (edited) config; we only override run-specific
# knobs here. Runs in a subprocess (fresh torch, no restart).
CMD = (f"python scripts/run_experiment.py --config {CONFIG} "
       f"checkpoint.output_dir={OUT_DIR} checkpoint.save_last_freq={SAVE_FREQ} "
       f"training.epochs={EPOCHS}")
if RESUME_CKPT:
    CMD += f" checkpoint.resume_from={RESUME_CKPT}"
print(CMD, "\n" + "="*70)
!{CMD}

## 7. Evaluate  *(comment out the whole cell to skip)*

In [ ]:
import glob, os

# Find the checkpoint to evaluate: this session's OUT_DIR first, then any
# attached notebook output (Save Version -> attach it as an input). Searches
# instead of hardcoding a notebook hash, so nothing needs editing after the
# first Save Version.
cands = []
for root in (os.path.dirname(OUT_DIR), "/kaggle/input"):
    for which in ("checkpoint-best.pt", "checkpoint-last.pt"):
        cands += sorted(glob.glob(f"{root}/**/{RUN}/{which}", recursive=True))
assert cands, f"no checkpoint found for {RUN} -- train first, or attach the saved Version"
EVAL_CKPT = cands[0]
EVAL_OUT  = f"/kaggle/working/{RUN}/eval"
print("eval ckpt:", EVAL_CKPT)

EVAL_CMD = (f"python scripts/evaluate.py --config {CONFIG} --checkpoint {EVAL_CKPT} "
            f"--n_samples 10000 --batch_size 200 --ema 1 "
            f"--cfg_scale 2.5 --cfg_interval 0.1 1.0 --out_dir {EVAL_OUT}")
print(EVAL_CMD, "\n" + "="*70)
!PYTHONPATH=. {EVAL_CMD}

## Notes
- Checkpoints (`-last`/`-best`/`-ep###`) with both EMA copies + optimizer are in `OUT_DIR` under `/kaggle/working`, so a resumed run equals a continuous one.
- **Resuming, also on another Kaggle account:** attach the previous session's notebook output, or upload its `checkpoint-last.pt` as a dataset and attach that. The settings cell finds it, keeps only checkpoints of this exact model (this model has the 3 token slots `incontext_pos_embed`, so a no-gate baseline checkpoint never matches), and resumes from the newest epoch. Check its `resume :` line before training.
- **Data:** the Tiny-ImageNet cell uses an attached dataset first, then Stanford, then the SJTU mirror of the identical zip (MD5-checked). To skip downloads for good, upload `tiny-imagenet-200.zip` once as your own Kaggle dataset and attach it.
- Keep `EPOCHS`, `cfg_scale`, `DATA_DIR`, seed and sampler identical to the other runs.
- Comparisons: **this vs Spatial no gate (93.95)** = what the in-context class tokens add to Spatial-Mamba; **this vs JiM + 1 class token (89.20)** = Spatial-Mamba vs SS2D, both with in-context class conditioning.
- Cost: 3 extra scan positions (+4.7%); the verify cell prints the it/s against the no-gate run.